In [9]:
import numpy as np
import pickle
import itertools
def has_hook_error(z, x):
    """
    For each Z-check row, and each possible time window, checks if there exists
    a time window such that, for the pattern of active X-couplings (syndrome), all X-syndromes
    'fall through' (no single qubit is uniquely flagged by an X-check) -- suggesting a "hook error" risk.
    
    Returns a list of (row, twindow) pairs where a hook error is detected.
    """
    hook_error_locations = []
    for row in z:
        for twindow in range(1, 4):
            mask = row > twindow
            xsyndrome = np.zeros((x.shape[0], x.shape[1]), dtype=bool)
            for j in range(x.shape[0]):
                masked = (x[j] > 0) & mask
                xsyndrome[j] = masked
            # A "hook error" occurs if for this twindow, no X-check covers only one qubit in the masked locations (i.e., all have 0 or >1 overlap)
            fallthrough = True
            for j in range(xsyndrome.shape[0]):
                if np.sum(xsyndrome[j]) == 1 or np.sum(xsyndrome[j]) == 3:
                    fallthrough = False
                    break
            if fallthrough:
                hook_error_locations.append((row.copy(), twindow))
    return hook_error_locations

def checkvalidschedule(schedule):
    for row in schedule:
        seen = set()
        for val in row:
            if val != 0:
                if val in seen:
                    return False
                seen.add(val)
    for col_idx in range(schedule.shape[1]):
        seen = set()
        for row_idx in range(schedule.shape[0]):
            val = schedule[row_idx][col_idx]
            if val != 0:
                if val in seen:
                    return False
                seen.add(val)
    return True

def randomize_schedule(schedule, max_attempts=1000):
    copyschedule = schedule.copy()
    if max_attempts == 0:
        return None
    for i in range(len(copyschedule)):
        nonzeros = list(range(1, max(copyschedule[i]) + 1)) if max(copyschedule[i]) != 1 else list(range(1, 7))
        np.random.shuffle(nonzeros)
        for j in range(len(copyschedule[i])):
            if copyschedule[i][j] != 0:
                copyschedule[i][j] = nonzeros.pop()
    if checkvalidschedule(copyschedule):
        return copyschedule, max_attempts
    else:
        return randomize_schedule(copyschedule, max_attempts - 1)

In [5]:
defaultxschedule = np.load('hx.npy')[0:7]
defaultzschedule = np.load('hz.npy')[0:7]
print(defaultxschedule)
print(defaultzschedule)

[[1 0 1 1 0 0 0 0 0 1 1 0 0 0 0 1 0 0]
 [1 1 0 0 1 0 0 0 0 0 1 1 0 0 0 0 1 0]
 [0 1 1 0 0 1 0 0 0 1 0 1 0 0 0 0 0 1]
 [0 0 0 1 0 1 1 0 0 1 0 0 1 1 0 0 0 0]
 [0 0 0 1 1 0 0 1 0 0 1 0 0 1 1 0 0 0]
 [0 0 0 0 1 1 0 0 1 0 0 1 1 0 1 0 0 0]
 [1 0 0 0 0 0 1 0 1 0 0 0 1 0 0 1 1 0]]
[[1 0 1 1 0 0 0 0 0 1 1 0 0 0 0 1 0 0]
 [1 1 0 0 1 0 0 0 0 0 1 1 0 0 0 0 1 0]
 [0 1 1 0 0 1 0 0 0 1 0 1 0 0 0 0 0 1]
 [0 0 0 1 0 1 1 0 0 1 0 0 1 1 0 0 0 0]
 [0 0 0 1 1 0 0 1 0 0 1 0 0 1 1 0 0 0]
 [0 0 0 0 1 1 0 0 1 0 0 1 1 0 1 0 0 0]
 [1 0 0 0 0 0 1 0 1 0 0 0 1 0 0 1 1 0]]


In [45]:
newzschedule, n = randomize_schedule(defaultzschedule, 10000)
print(10000 - n)
print(newzschedule)
newxschedule, n = randomize_schedule(defaultxschedule, 10000)
print(10000 - n)
print(newxschedule)


466
[[1 0 4 6 0 0 0 0 0 5 3 0 0 0 0 2 0 0]
 [4 1 0 0 5 0 0 0 0 0 2 3 0 0 0 0 6 0]
 [0 3 1 0 0 5 0 0 0 4 0 2 0 0 0 0 0 6]
 [0 0 0 1 0 3 5 0 0 2 0 0 6 4 0 0 0 0]
 [0 0 0 5 4 0 0 3 0 0 1 0 0 6 2 0 0 0]
 [0 0 0 0 2 6 0 0 3 0 0 4 1 0 5 0 0 0]
 [6 0 0 0 0 0 2 0 5 0 0 0 4 0 0 3 1 0]]
694
[[2 0 3 6 0 0 0 0 0 4 5 0 0 0 0 1 0 0]
 [5 1 0 0 4 0 0 0 0 0 6 3 0 0 0 0 2 0]
 [0 2 5 0 0 6 0 0 0 3 0 4 0 0 0 0 0 1]
 [0 0 0 5 0 4 2 0 0 1 0 0 3 6 0 0 0 0]
 [0 0 0 4 1 0 0 2 0 0 3 0 0 5 6 0 0 0]
 [0 0 0 0 2 5 0 0 3 0 0 1 6 0 4 0 0 0]
 [3 0 0 0 0 0 4 0 6 0 0 0 2 0 0 5 1 0]]


In [51]:
#hook_errors = has_hook_error(defaultzschedule, defaultxschedule)
hook_errors = has_hook_error(newzschedule, newxschedule)
if hook_errors:
    print("Hook errors detected at:")
    for row, twindow in hook_errors:
        print("Z row:", row, "at twindow:", twindow)
else:
    print("No hook errors detected.")


No hook errors detected.


In [55]:
z832 = np.asarray([
    [1, 2, 3, 4, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 2, 3, 4],
    [2, 3, 0, 0, 4, 1, 0, 0],
    [3, 0, 4, 0, 2, 0, 1, 0]
])
x832 = np.asarray([
    [1, 2, 3, 4, 5, 6, 7, 8]
])
hook_errors = has_hook_error(z832, x832)
print(hook_errors)

[(array([1, 2, 3, 4, 0, 0, 0, 0]), 2), (array([0, 0, 0, 0, 1, 2, 3, 4]), 2), (array([2, 3, 0, 0, 4, 1, 0, 0]), 2), (array([3, 0, 4, 0, 2, 0, 1, 0]), 2)]
